# 01 — Ingestão e Diagnóstico dos Dados Heterogêneos

Etapa 1 do procedimento metodológico (Seção 3.4). Objetivos deste notebook:

1. Carregar a base e documentar o dicionário de variáveis (tipo, faltantes, cardinalidade).
2. Caracterizar a variável-alvo (`no_show`): desbalanceamento e deriva temporal.
3. Identificar vazamento de dados, inconsistências e padrões de ausência.
4. Verificar a estrutura de pacientes recorrentes (impacto na validação cruzada).
5. Diagnosticar as variáveis contínuas (escala, assimetria, outliers) e as associações univariadas com o no-show.

Nenhuma transformação aplicada aqui altera a base: as decisões ficam registradas no fim e são implementadas em `src/noshow/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from noshow.data import load_raw, add_patient_proxy, TARGET, LEAKAGE_COLS, PATIENT_KEY, RAW_PATH
from noshow.weather import fix_weather

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

# Paleta: uma cor para magnitude, uma segunda só quando há duas classes; texto/grade neutros.
BLUE, ORANGE, GRAY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

## 1. Carga e dicionário de variáveis

In [ ]:
raw = pd.read_csv(RAW_PATH)  # versão sem nenhum tratamento, para comparação
df = add_patient_proxy(load_raw())
print(f"{df.shape[0]:,} registros × {raw.shape[1]} colunas originais")
print("Período:", df.appointment_date.min().date(), "a", df.appointment_date.max().date())

In [ ]:
TIPOS = {
    "specialty": "categórica", "appointment_time": "temporal", "gender": "categórica",
    "appointment_date": "temporal", "no_show": "alvo", "no_show_reason": "texto (vazamento)",
    "disability": "categórica", "date_of_birth": "temporal", "entry_service_date": "temporal",
    "city": "categórica", "icd": "categórica", "appointment_month": "temporal",
    "appointment_year": "temporal", "appointment_shift": "temporal", "age": "contínua",
    "under_12_years_old": "binária", "over_60_years_old": "binária",
    "patient_needs_companion": "binária", "average_temp_day": "contínua",
    "average_rain_day": "contínua", "max_temp_day": "contínua", "max_rain_day": "contínua",
    "rainy_day_before": "binária", "storm_day_before": "binária",
    "rain_intensity": "categórica (ordinal)", "heat_intensity": "categórica (ordinal)",
}
GRUPO = {  # categorias de fatores da Seção 2.1.4
    "specialty": "contextual", "appointment_time": "temporal", "gender": "demográfico",
    "appointment_date": "temporal", "disability": "demográfico", "date_of_birth": "demográfico",
    "entry_service_date": "comportamental", "city": "demográfico", "icd": "demográfico",
    "appointment_month": "temporal", "appointment_year": "temporal", "appointment_shift": "temporal",
    "age": "demográfico", "under_12_years_old": "demográfico", "over_60_years_old": "demográfico",
    "patient_needs_companion": "contextual", "average_temp_day": "contextual (clima)",
    "average_rain_day": "contextual (clima)", "max_temp_day": "contextual (clima)",
    "max_rain_day": "contextual (clima)", "rainy_day_before": "contextual (clima)",
    "storm_day_before": "contextual (clima)", "rain_intensity": "contextual (clima)",
    "heat_intensity": "contextual (clima)",
}
cols = raw.columns
dicionario = pd.DataFrame({
    "tipo": [TIPOS[c] for c in cols],
    "grupo (Seção 2.1.4)": [GRUPO.get(c, "—") for c in cols],
    "faltantes_%": (df[cols].isna().mean() * 100).round(1).values,
    "valores_únicos": df[cols].nunique().values,
    "exemplo": [df[c].dropna().iloc[0] for c in cols],
}, index=cols)
dicionario

**Lacunas em relação ao projeto.** A base **não tem identificador de paciente** nem **data de marcação** do agendamento.
Por isso, não dá para calcular diretamente o tempo de espera fracionado (solicitação → autorização → execução,
Seções 1.2 e 2.1.4) nem o histórico de faltas. As alternativas possíveis são:

- `entry_service_date` → *tempo de vínculo com o serviço* (dias entre a entrada e a consulta);
- um **ID de paciente reconstruído** (Seção 5), que permite criar variáveis de histórico de comparecimento.

## 2. Variável-alvo: desbalanceamento e deriva temporal

In [ ]:
taxa = df[TARGET].mean()
print(f"No-show: {df[TARGET].sum():,} de {len(df):,} ({taxa:.1%}) — razão {1/taxa - 1:.1f}:1 a favor do comparecimento")

In [ ]:
por_ano = df.groupby("appointment_year")[TARGET].agg(taxa="mean", registros="size")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.bar(por_ano.index, por_ano.taxa, color=BLUE, width=0.6)
ax.axhline(taxa, color=GRAY, lw=1, ls="--")
ax.text(por_ano.index.min() - 0.4, taxa, f"média geral {taxa:.1%}", va="bottom", fontsize=8, color=INK)
for x, y in por_ano.taxa.items():
    ax.text(x, y, f"{y:.1%}", ha="center", va="bottom", fontsize=8, color=INK)
ax.set_title("Taxa de no-show por ano")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1, decimals=0))

ax = axes[1]
ax.bar(por_ano.index, por_ano.registros, color=BLUE, width=0.6)
ax.set_title("Volume de agendamentos por ano")
plt.tight_layout(); plt.show()
por_ano

In [ ]:
mensal = df.set_index("appointment_date")[TARGET].resample("MS").agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(mensal.index, mensal["mean"], color=BLUE, lw=2)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-12-31"), color=GRAY, alpha=0.12)
ax.text(pd.Timestamp("2020-03-15"), ax.get_ylim()[1] * 0.92, "pandemia de COVID-19", fontsize=8, color=INK)
ax.set_title("Taxa mensal de no-show")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1, decimals=0))
plt.tight_layout(); plt.show()
print("Meses com menos de 100 registros:", (mensal["size"] < 100).sum())

A taxa de no-show **quadruplica** entre 2016 e 2022, e o volume anual varia muito.
Isso é **deriva temporal (*concept drift*)**: um k-fold aleatório mistura anos e mede um desempenho que um
sistema em produção não teria. O ano precisa entrar como variável e/ou a validação precisa respeitar o tempo
(ver decisões no fim).

**Cobertura irregular:** há meses sem nenhum registro (2018-09, 2019-01 e quase todo o 1º semestre de 2020) e
meses com menos de 100 agendamentos durante a pandemia. Já 2022 vai só até fevereiro. As taxas de 2020 e 2022
vêm de menos de mil registros cada e devem ser lidas com cautela.

## 3. Vazamento de dados: `no_show_reason`

In [ ]:
pd.crosstab(df.no_show_reason.notna().map({True: "motivo preenchido", False: "motivo vazio"}),
            df[TARGET].map({0: "compareceu", 1: "no-show"}), margins=True)

In [ ]:
print("Motivos mais frequentes:")
df.no_show_reason.str.lower().value_counts().head(10)

O motivo só existe **depois** da falta: 1.677 dos 1.737 registros preenchidos são no-show (97%).
Usar essa coluna como preditor seria vazamento direto. Ela **sai** do pipeline (`LEAKAGE_COLS`).
Ainda assim, pode ser útil para a discussão qualitativa dos resultados.

## 4. Faltantes, inconsistências e duplicatas

In [ ]:
bloco_demografico = ["date_of_birth", "entry_service_date", "city", "age", "disability"]
sem_demografia = df[bloco_demografico].isna().all(axis=1)
print(f"Registros sem nenhum dado demográfico: {sem_demografia.sum():,} ({sem_demografia.mean():.1%})")

ausencia = pd.DataFrame({
    col: df.groupby(df[col].isna())[TARGET].mean().rename({False: "presente", True: "faltante"})
    for col in ["specialty", "age", "entry_service_date", "icd", "average_temp_day"]
}).T
ausencia["n_faltantes"] = [df[c].isna().sum() for c in ausencia.index]
ausencia["chi2_p"] = [stats.chi2_contingency(pd.crosstab(df[c].isna(), df[TARGET]))[1] for c in ausencia.index]
ausencia.style.format({"presente": "{:.1%}", "faltante": "{:.1%}", "chi2_p": "{:.1e}", "n_faltantes": "{:,}"})

A ausência **não é aleatória** (*missing not at random*): a taxa de no-show muda bastante quando o campo está vazio.
Descartar linhas ou imputar pela média apagaria esse sinal. Por isso, nas variáveis categóricas, **"faltante" vira uma
categoria própria**, o que se encaixa bem em uma Rede Bayesiana discreta.

In [ ]:
erros_data = {}
for c in ["entry_service_date", "date_of_birth"]:
    original = raw[c].astype("string").str.strip().replace("", pd.NA)
    falhou = original.notna() & df[c].isna()
    erros_data[c] = original[falhou].value_counts().head(6)
    print(f"{c}: {falhou.sum()} datas inválidas na origem → convertidas em NaN")
    print(erros_data[c].to_string(), "\n")

In [ ]:
inconsistencias = {
    "gender = 'I' (indeterminado)": (df.gender == "I").sum(),
    "disability só com espaço em branco": (raw.disability.astype(str).str.strip() == "").sum(),
    "specialty = 'sem especialidade'": (df.specialty == "sem especialidade").sum(),
    "consulta antes da entrada no serviço": ((df.appointment_date - df.entry_service_date).dt.days < 0).sum(),
    "idade > 100": (df.age > 100).sum(),
    "consulta no fim de semana": (df.appointment_date.dt.dayofweek >= 5).sum(),
    "linhas duplicadas (exatas)": raw.duplicated().sum(),
}
pd.Series(inconsistencias, name="registros").to_frame()

In [ ]:
dup = raw.duplicated(keep=False)
print(f"Duplicatas: {dup.mean():.1%} das linhas | no-show {raw[dup].no_show.eq('yes').mean():.1%} "
      f"| sem demografia {raw[dup].age.isna().mean():.0%}")

As duplicatas exatas são, em geral, registros **sem dados demográficos** no mesmo dia, horário e especialidade.
Podem ser pacientes diferentes atendidos em grupo, que os campos disponíveis não conseguem distinguir.
**Decisão: manter** (remover enviesaria a amostra para comparecimento) e registrar como limitação.

## 5. Pacientes recorrentes (ID reconstruído)

A base não traz ID, mas a combinação `date_of_birth`, `gender`, `city`, `entry_service_date` e `disability` é estável para um mesmo paciente.
Tratamos essa combinação como um **identificador aproximado** (`patient_id`).

In [ ]:
com_id = df.patient_id.notna()
consultas = df.loc[com_id].groupby("patient_id").size()
print(f"{com_id.mean():.1%} dos registros têm ID reconstruído → {consultas.size:,} pacientes")
print(consultas.describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(consultas, bins=40, color=BLUE, edgecolor="white", linewidth=1)
ax.set_title("Agendamentos por paciente (ID reconstruído)")
ax.set_xlabel("agendamentos"); ax.set_ylabel("pacientes")
plt.tight_layout(); plt.show()

In [ ]:
por_paciente = df.loc[com_id].groupby("patient_id")[TARGET].agg(["sum", "size"])
ordenado = por_paciente.sort_values("sum", ascending=False)
top10 = ordenado.head(int(len(ordenado) * 0.10))["sum"].sum() / por_paciente["sum"].sum()
print(f"Pacientes com ao menos 1 no-show: {(por_paciente['sum'] > 0).mean():.1%}")
print(f"Os 10% de pacientes que mais faltam concentram {top10:.1%} de todos os no-shows")

**Implicações:**

1. O comportamento de faltar está **concentrado em poucos pacientes**. Isso confirma o histórico de faltas como
   preditor central (Seção 2.1.4). Essa variável pode ser construída com o `patient_id`, contando **somente
   consultas anteriores** à data de cada agendamento.
2. Cada paciente aparece em média em dezenas de agendamentos. Em um k-fold aleatório, o mesmo paciente fica no
   treino e no teste, e atributos como data de nascimento + cidade funcionam como "impressão digital". Isso gera
   **vazamento por paciente**. A validação deve usar `StratifiedGroupKFold` por `patient_id` (ou divisão temporal).

## 6. Variáveis contínuas: escala, assimetria e outliers

In [ ]:
df["dias_em_servico"] = (df.appointment_date - df.entry_service_date).dt.days  # prévia; formalizada no notebook 02
continuas = ["age", "dias_em_servico", "average_temp_day", "max_temp_day", "average_rain_day", "max_rain_day"]

def resumo(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    out = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()
    return pd.Series({"min": s.min(), "mediana": s.median(), "max": s.max(), "média": s.mean(),
                      "desvio": s.std(), "assimetria": s.skew(), "outliers_IQR_%": out * 100,
                      "zeros_%": (s == 0).mean() * 100})

pd.DataFrame({c: resumo(df[c].dropna()) for c in continuas}).T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.flat, continuas):
    for cls, cor, nome in [(0, BLUE, "compareceu"), (1, ORANGE, "no-show")]:
        ax.hist(df.loc[df[TARGET] == cls, c].dropna(), bins=40, density=True, histtype="step",
                lw=2, color=cor, label=nome)
    ax.set_title(c); ax.set_yticks([])
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Distribuição das variáveis contínuas por classe (densidade)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

- As escalas são bem diferentes (idade 2–110, dias em serviço até ~2.700, chuva quase sempre zero). Além disso,
  chuva e dias em serviço têm assimetria forte e muitos outliers, o que **justifica o RobustScaler** como candidato
  principal **para a distância do SMOTENC**.
- **Atenção metodológica:** a Rede Bayesiana discreta só enxerga as faixas após a discretização. Escalonadores
  lineares (MinMax/Standard/Robust) **não mudam essas faixas** (ver célula abaixo). O efeito do escalonamento
  sobre o modelo vem, portanto, apenas do SMOTENC.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer

x = df[["dias_em_servico"]].dropna().to_numpy()
for estrategia in ["quantile", "uniform"]:
    disc = lambda m: KBinsDiscretizer(n_bins=5, encode="ordinal", strategy=estrategia,
                                      quantile_method="averaged_inverted_cdf").fit_transform(m)
    base = disc(x)
    iguais = {nome: np.array_equal(base, disc(sc.fit_transform(x)))
              for nome, sc in [("MinMax", MinMaxScaler()), ("Standard", StandardScaler()), ("Robust", RobustScaler())]}
    print(f"discretização '{estrategia}': faixas idênticas às dos dados brutos? {iguais}")

## 7. Associação univariada com o no-show

Antes de medir as associações, o clima é corrigido com `fix_weather` (detalhes e validação na EDA, notebook 00).
Nos 1.016 agendamentos sem medição (32 dias), a base gravou "frio intenso" e "sem chuva", e as colunas
`rainy_day_before`/`storm_day_before` só variam nesses dias. A correção remove esses artefatos e preenche a
temperatura com o Open-Meteo (r = 0,98 contra os dias medidos); a chuva desses dias fica ausente.

In [ ]:
df = fix_weather(df)
print(df.clima_fonte.value_counts().to_dict())

In [ ]:
def cramers_v(x, y):
    tabela = pd.crosstab(x, y)
    chi2, p, _, _ = stats.chi2_contingency(tabela)
    n = tabela.to_numpy().sum()
    return np.sqrt(chi2 / (n * (min(tabela.shape) - 1))), p

categoricas = ["specialty", "gender", "disability", "city", "icd", "appointment_month", "appointment_year",
               "appointment_shift", "appointment_time", "under_12_years_old", "over_60_years_old",
               "patient_needs_companion", "rain_intensity", "heat_intensity"]
df["dia_semana"] = df.appointment_date.dt.day_name()
categoricas.append("dia_semana")

linhas = []
for c in categoricas:
    v, p = cramers_v(df[c].astype("string").fillna("faltante"), df[TARGET])
    linhas.append({"variável": c, "tipo": "categórica", "efeito": v, "medida": "V de Cramér", "p": p})
for c in continuas:
    a, b = df.loc[df[TARGET] == 1, c].dropna(), df.loc[df[TARGET] == 0, c].dropna()
    u, p = stats.mannwhitneyu(a, b)
    linhas.append({"variável": c, "tipo": "contínua", "efeito": abs(2 * u / (len(a) * len(b)) - 1),
                   "medida": "|rank-biserial|", "p": p})

associacao = pd.DataFrame(linhas).sort_values("efeito", ascending=False).reset_index(drop=True)
associacao.style.format({"efeito": "{:.3f}", "p": "{:.1e}"}).bar(subset="efeito", color="#c9dcf3")

Com ~50 mil registros, quase tudo dá *p* < 0,05. Por isso, a ordenação usa o **tamanho do efeito**, não o *p*-valor.
Essa associação é **marginal** (uma variável por vez): o Manto de Markov (notebook 03) vai dizer o que continua
relevante quando as outras variáveis são levadas em conta.

In [ ]:
mostrar = ["specialty", "disability", "appointment_shift", "dia_semana", "rain_intensity", "heat_intensity"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5))
for ax, c in zip(axes.flat, mostrar):
    g = (df.assign(_c=df[c].astype("string").fillna("faltante"))
           .groupby("_c")[TARGET].agg(["mean", "size"]).sort_values("mean"))
    ax.barh(g.index, g["mean"], color=BLUE, height=0.6)
    ax.axvline(taxa, color=GRAY, lw=1, ls="--")
    for i, (m, n) in enumerate(zip(g["mean"], g["size"])):
        ax.text(m, i, f" {m:.1%} (n={n:,})", va="center", fontsize=7, color=INK)
    ax.set_title(c); ax.set_xlim(0, g["mean"].max() * 1.6)
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1, decimals=0))
    ax.grid(axis="y", visible=False)
fig.suptitle("Taxa de no-show por categoria (linha tracejada = média geral)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Síntese e decisões para o pipeline

In [ ]:
print(f"Registros: {len(df):,} | no-show: {taxa:.1%} | pacientes reconstruídos: {consultas.size:,}")
print("Variáveis com maior efeito marginal:", ", ".join(associacao.head(6)["variável"]))

| # | Achado | Decisão |
|---|--------|---------|
| 1 | `no_show_reason` só existe após a falta | Remover (vazamento) |
| 2 | No-show sobe de 4% (2016) para 17% (2022) | Ano como variável e validação ciente do tempo; discutir COVID |
| 3 | Classe minoritária ≈ 9,7% | SMOTENC apenas no fold de treino (`imblearn.Pipeline`) |
| 4 | Faltantes informativos (MNAR) | Categoria "faltante" explícita; sem descarte de linhas |
| 5 | Sem ID nem data de marcação | ID reconstruído; `dias_em_servico` no lugar do tempo de espera fracionado |
| 6 | Pacientes com dezenas de consultas | `StratifiedGroupKFold` por `patient_id`; histórico calculado só com o passado |
| 7 | Contínuas assimétricas, com outliers | RobustScaler para o SMOTENC; discretização por quantis para a RB |
| 8 | Escalonamento linear não altera a discretização | Ablação no notebook 04; ajustar a justificativa da Seção 2.3.2 |
| 9 | Datas com erro de digitação na origem | NaN + registro como limitação |
| 10 | Clima ausente gravado como "frio intenso"/"sem chuva"; `rainy/storm_day_before` são artefatos | `fix_weather`: remove artefatos e preenche a temperatura dos 32 dias com o Open-Meteo |

**Próximo notebook:** `02_engenharia_variaveis.ipynb`, que vai criar o histórico de faltas, o tempo de vínculo,
as variáveis de calendário e a discretização.